# Module 2: Accident Classification & Root Cause Analysis

Evaluate multi-class accident classifiers and use SHAP/gradient attribution
to identify which sensors and time windows drive each diagnosis.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from pathlib import Path

from src.data.dataset import NPPADWindowDataset
from src.models.diagnosis import build_classifier
from src.evaluation.classification_metrics import evaluate_classifier
from src.models.diagnosis.interpretability import gradient_based_attribution
from src.utils.config import Config
from data.scripts.preprocess import ACCIDENT_TYPES

sns.set_theme(style='whitegrid')
CHECKPOINT_DIR = Path('../checkpoints')

inv_map = {v: k for k, v in ACCIDENT_TYPES.items()}
FEATURE_NAMES = [
    'P', 'TAVG', 'THA', 'THB', 'TCA', 'TCB', 'WRCA', 'WRCB',
    'PSGA', 'PSGB', 'WFWA', 'WFWB', 'WSTA', 'WSTB', 'VOL', 'LVPZ',
    'VOID', 'WLR', 'WUP', 'HUP', 'HLW', 'WHPI', 'WECS', 'QMWT',
    'LSGA', 'LSGB', 'QMGA', 'QMGB', 'NSGA', 'NSGB', 'TBLD', 'WTRA',
    'WTRB', 'TSAT', 'QRHR', 'LVCR', 'SCMA', 'SCMB', 'FRCL', 'PRB',
    'PRBA', 'TRB', 'LWRB', 'DNBR', 'QFCL', 'WBK', 'WSPY', 'WCSP',
    'HTR', 'MH2', 'CNH2', 'RHBR', 'RHMT', 'RHFL', 'RHRD', 'RH',
    'PWNT', 'PWR', 'TFSB', 'TFPK', 'TF', 'TPCT', 'WCFT', 'WLPI',
    'WCHG', 'RM1', 'RM2', 'RM3', 'RM4', 'RC87', 'RC131', 'STRB',
    'STSG', 'STTB', 'RBLK', 'SGLK', 'DTHY', 'DWB', 'WRLA', 'WRLB',
    'WLD', 'MBK', 'EBK', 'TKLV', 'FRZR', 'TDBR', 'MDBR', 'MCRT',
    'MGAS', 'TCRT', 'TSLP', 'PPM', 'RRCA', 'RRCB', 'RRCO', 'WFLB',
]

## 1. Load Models and Test Data

In [ ]:
test_ds = NPPADWindowDataset('../data/processed/test.pt')
test_ds.labels = test_ds.accident_types  # Use accident type labels
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

model_configs = [
    ('cnn_classifier', '../configs/cnn_classifier.yaml'),
    ('lstm_classifier', '../configs/lstm_classifier.yaml'),
    ('transformer_classifier', '../configs/transformer_classifier.yaml'),
]

models = {}
results = {}
for name, cfg_path in model_configs:
    config = Config.from_yaml(cfg_path)
    model = build_classifier(config)
    ckpt = torch.load(CHECKPOINT_DIR / f'{name}_best.pt', weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    models[name] = model
    
    r = evaluate_classifier(model, test_loader)
    results[name] = r
    print(f'{name}: acc={r["accuracy"]:.3f}, F1_macro={r["f1_macro"]:.3f}, F1_weighted={r["f1_weighted"]:.3f}')

## 2. Confusion Matrix (Best Model)

In [ ]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_r = results[best_name]
cm = best_r['confusion_matrix']

# Get class labels present in test set
present_classes = sorted(set(best_r['labels'].tolist()) | set(best_r['predictions'].tolist()))
class_labels = [inv_map.get(c, str(c)) for c in present_classes]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_labels, yticklabels=class_labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix - {best_name} (acc={best_r["accuracy"]:.3f})')
plt.tight_layout()
plt.show()

## 3. Per-Class Performance Comparison

In [ ]:
import pandas as pd

# Extract per-class F1 for each model
f1_data = {}
for name, r in results.items():
    report = r['classification_report']
    f1_data[name] = {}
    for key, vals in report.items():
        if isinstance(vals, dict) and 'f1-score' in vals and key.isdigit():
            class_name = inv_map.get(int(key), f'Class {key}')
            f1_data[name][class_name] = vals['f1-score']

df_f1 = pd.DataFrame(f1_data)
df_f1.plot(kind='barh', figsize=(12, 8))
plt.xlabel('F1 Score')
plt.title('Per-Class F1 Score by Model')
plt.legend(title='Model')
plt.tight_layout()
plt.show()

## 4. Root Cause Analysis: Gradient Attribution

Which sensors contribute most to each accident diagnosis?

In [ ]:
# Use best model for interpretability
best_model = models[best_name]

# Get a batch of correctly classified samples for each accident type
test_windows = test_ds.windows
test_labels = test_ds.accident_types

# Compute attributions for selected accident types
target_classes = [1, 3, 8, 11]  # LOCA, SLBIC, LLB, LR
target_names = [inv_map[c] for c in target_classes]

fig, axes = plt.subplots(len(target_classes), 1, figsize=(16, 4 * len(target_classes)))

for ax, cls_idx, cls_name in zip(axes, target_classes, target_names):
    mask = test_labels == cls_idx
    if mask.sum() == 0:
        continue
    
    samples = test_windows[mask][:20]  # First 20 samples of this class
    attrs = gradient_based_attribution(best_model, samples, target_class=cls_idx)
    
    # Average absolute attribution across samples and time
    feat_importance = attrs.abs().mean(dim=(0, 1)).numpy()
    
    # Top 15 features
    top_idx = np.argsort(feat_importance)[-15:]
    top_names = [FEATURE_NAMES[i] for i in top_idx]
    top_vals = feat_importance[top_idx]
    
    ax.barh(top_names, top_vals)
    ax.set_xlabel('Mean |Attribution|')
    ax.set_title(f'{cls_name} - Top 15 Contributing Sensors')

plt.suptitle('Root Cause Analysis: Sensor Attribution by Accident Type', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Temporal Attribution

When in the window does each accident type become identifiable?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for cls_idx, cls_name in zip(target_classes, target_names):
    mask = test_labels == cls_idx
    if mask.sum() == 0:
        continue
    
    samples = test_windows[mask][:20]
    attrs = gradient_based_attribution(best_model, samples, target_class=cls_idx)
    
    # Mean absolute attribution per timestep
    temporal = attrs.abs().mean(dim=(0, 2)).numpy()
    ax.plot(temporal, label=cls_name, alpha=0.8)

ax.set_xlabel('Timestep in Window')
ax.set_ylabel('Mean |Attribution|')
ax.set_title('Temporal Attribution - When Accidents Become Identifiable')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Summary Metrics Table

In [ ]:
summary = pd.DataFrame({
    name: {
        'Accuracy': r['accuracy'],
        'F1 (macro)': r['f1_macro'],
        'F1 (weighted)': r['f1_weighted'],
        'Precision (macro)': r['precision_macro'],
        'Recall (macro)': r['recall_macro'],
    }
    for name, r in results.items()
}).T

summary.index.name = 'Model'
print(summary.to_string(float_format='%.4f'))